In [ ]:
#Builds rolling team-form features, then splits games by season

In [1]:
# loads the panda library and gives it nickname pd
import pandas as pd
import nflreadpy as nfl

seasons = range(2015, 2026)
games = nfl.load_schedules(seasons=seasons).to_pandas()
print(games.shape)

# picks out gameday column, and converts it from plaintext to a real date.
games["gameday"] = pd.to_datetime(games["gameday"])

#.sort_values("gameday") puts the games in date order. 
#.reset_index(drop=True) resets the index to be 0,1,2,3,... instead of the original index.
games = games.sort_values("gameday").reset_index(drop=True)

# Compares home_score and away_score, and creates a new column called home_win. 
# If home_score is greater than away_score, home_win will be 1, otherwise it will be 0.
games["home_win"] = (games["home_score"] > games["away_score"]).astype(int)


(3028, 46)


In [2]:
#This step turns a table with one row per game into a table with two rows per game, one 
# for the home team and one for the away team.

#Selects the columns we want for the home team, and renames them to be more generic. 
#.copy() makes a copy of the data so that we can modify it without changing the original dataframe.
home = games[["game_id", "gameday", "home_team", "home_score", "away_score"]].copy()

# Renames the columns to be more generic, so that we can combine the home and away data later.
home.columns = ["game_id", "gameday", "team", "pts_for", "pts_against"]

# Creates a new column called is_home, and sets it to 1 for all rows in the home dataframe.
home["is_home"] = 1

#Displays the rows of the home dataframe.
home.head()

# Same thing, but just for the away columns. "is_home" is set to 0 for all rows in the away dataframe.
away = games[["game_id", "gameday", "away_team", "away_score", "home_score"]].copy()
away.columns = ["game_id", "gameday", "team", "pts_for", "pts_against"]
away["is_home"] = 0

#Stacks the table vertically. .sort_values(["team", "gameday"]) sorts the table by team and then by date. 
#.reset_index(drop=True) resets the index to be 0,1,2,3,... instead of the original index.
team_games = pd.concat([home, away]).sort_values(["team", "gameday"]).reset_index(drop=True)

# Creates new column called win, which is 1 if the team won the game and 0 if they lost.
team_games["win"] = (team_games["pts_for"] > team_games["pts_against"]).astype(int)

# Creates new column called margin, which is the difference between points scored and points allowed.
team_games["margin"] = team_games["pts_for"] - team_games["pts_against"]
team_games.head(10)

,game_id,gameday,team,pts_for,pts_against,is_home,win,margin
0,2015_01_NO_ARI,2015-09-13,ARI,31,19,1,1,12
1,2015_02_ARI_CHI,2015-09-20,ARI,48,23,0,1,25
2,2015_03_SF_ARI,2015-09-27,ARI,47,7,1,1,40
3,2015_04_STL_ARI,2015-10-04,ARI,22,24,1,0,-2
4,2015_05_ARI_DET,2015-10-11,ARI,42,17,0,1,25
5,2015_06_ARI_PIT,2015-10-18,ARI,13,25,0,0,-12
6,2015_07_BAL_ARI,2015-10-26,ARI,26,18,1,1,8
7,2015_08_ARI_CLE,2015-11-01,ARI,34,20,0,1,14
8,2015_10_ARI_SEA,2015-11-15,ARI,39,32,0,1,7
9,2015_11_CIN_ARI,2015-11-22,ARI,34,31,1,1,3


In [3]:
print(team_games.shape)
team_games["team"].nunique()
team_games[team_games["team"] == "KC"].head(3)

(6056, 8)


,game_id,gameday,team,pts_for,pts_against,is_home,win,margin
2819,2015_01_KC_HOU,2015-09-13,KC,27,20,0,1,7
2820,2015_02_DEN_KC,2015-09-17,KC,24,31,1,0,-7
2821,2015_03_KC_GB,2015-09-28,KC,28,38,0,0,-10


In [4]:
# STEP 3: gives every team-game row two numbers describing the team’s form before that game: its win rate 
# and its average margin over the previous 5 games.

# Names a variable named "g", and sets it equal to the team_games dataframe grouped by the "team" column. 
# This allows us to perform calculations on each team separately.
g = team_games.groupby("team")

# Creates a new column called win_pct_last5. g["win"] selects the "win" column from the grouped dataframe.
#  .transform() applies a function to each group and returns a new series with the same index as the original dataframe. 
# The lambda function takes a series s, shifts it by 1 (so that we don't include the current game in the calculation), 
# and then calculates the rolling mean over the last 5 games, with a minimum of 1 game required to calculate the mean.
team_games["win_pct_last5"] = g["win"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())
team_games["margin_last5"] = g["margin"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())

#Shows KC's last 8 games, with the new columns win_pct_last5 and margin_last5 included.
team_games[team_games["team"] == "KC"].head(8)

,game_id,gameday,team,pts_for,pts_against,is_home,win,margin,win_pct_last5,margin_last5
2819,2015_01_KC_HOU,2015-09-13,KC,27,20,0,1,7,NaN,NaN
2820,2015_02_DEN_KC,2015-09-17,KC,24,31,1,0,-7,1.000000,7.000000
2821,2015_03_KC_GB,2015-09-28,KC,28,38,0,0,-10,0.500000,0.000000
2822,2015_04_KC_CIN,2015-10-04,KC,21,36,0,0,-15,0.333333,-3.333333
2823,2015_05_CHI_KC,2015-10-11,KC,17,18,1,0,-1,0.250000,-6.250000
2824,2015_06_KC_MIN,2015-10-18,KC,10,16,0,0,-6,0.200000,-5.200000
2825,2015_07_PIT_KC,2015-10-25,KC,23,13,1,1,10,0.000000,-7.800000
2826,2015_08_DET_KC,2015-11-01,KC,45,10,1,1,35,0.200000,-4.400000


In [5]:
# STEP 4: the features currently live in the one-row-per-team table. 
# This cell copies them back onto the original one-row-per-game table, with separate home and away columns.

# Stores win_pct_last5 and margin_last5 in a list called feat_cols.
# This is useful for selecting these columns later.
feat_cols = ["win_pct_last5", "margin_last5"]

# Selects only 3 kinds of columns, the game ID, home/away flag,
# and the features. Then, the 2 lists are joined together.
feats = team_games[["game_id", "is_home"] + feat_cols]

# Keeps only the home team's rows, and removes the flag column
home_f = feats[feats["is_home"] == 1].drop(columns="is_home")
# Renames the feature columns to start with home_
home_f = home_f.rename(columns={c: f"home_{c}" for c in feat_cols})

# Same thing for away teams
away_f = feats[feats["is_home"] == 0].drop(columns="is_home")
away_f = away_f.rename(columns={c: f"away_{c}" for c in feat_cols})

games = games.merge(home_f, on="game_id").merge(away_f, on="game_id")
print(games.shape)





(3028, 51)


In [6]:
# Creates new columns that are the difference between the home and away features. 
# Displays a few row of the new columns (double brackets return a smaller table)
games["win_pct_diff"] = games["home_win_pct_last5"] - games["away_win_pct_last5"]
games["margin_diff"]  = games["home_margin_last5"] - games["away_margin_last5"]
games["rest_diff"]    = games["home_rest"] - games["away_rest"]
games[["home_team", "away_team", "win_pct_diff", "margin_diff", "rest_diff"]].head()

,home_team,away_team,win_pct_diff,margin_diff,rest_diff
0,NE,PIT,NaN,NaN,0
1,DAL,NYG,NaN,NaN,0
2,TB,TEN,NaN,NaN,0
3,DEN,BAL,NaN,NaN,0
4,SD,DET,NaN,NaN,0


In [7]:
# Removes games that are missing the win_pct_diff or margin_diff values. 
# This is important because we can't train a model on missing data.
games = games.dropna(subset=["win_pct_diff", "margin_diff"])

# Keeps games from 2022 and earlier
train = games[games["season"] <= 2022]

# Keeps games from 2023 and later
test  = games[games["season"] >= 2023]

In [8]:
# Counts how many missing values there are in the win_pct_diff, margin_diff, and rest_diff columns.
print(games[["win_pct_diff", "margin_diff", "rest_diff"]].isna().sum())

# Prints how many games are in the train and test sets
print(len(train), len(test))

# Prints the distinct seasons in the test set
print(test["season"].unique())

# Prints the average home win rate in the train and test sets.
print(train["home_win"].mean(), test["home_win"].mean())

kc = team_games[team_games["team"] == "KC"].reset_index(drop=True)
print(kc.loc[5, "win_pct_last5"], kc.loc[0:4, "win"].mean())
print(kc.loc[5, "margin_last5"], kc.loc[0:4, "margin"].mean())

win_pct_diff    0
margin_diff     0
rest_diff       0
dtype: int64
2154 855
[2023 2024 2025]
0.5468895078922934 0.5485380116959064
0.2 0.2
-5.2 -5.2


In [10]:
games.to_csv("../data/features.csv", index=False)